# Challenge - Starter notebook: skin lesions (DermaMNIST)

This notebook is your **starting point** for the challenge. We go through it together on day 4.
It does everything once, in the simplest way:
1. load the data and **look at it**,
2. train a **baseline CNN** (= "A" in the handout),
3. **evaluate** it on the validation images with the challenge metrics,
4. **export** the model as `model.pt` - this file is what you hand in.

Then it is your turn: improve the baseline (= "B") - ideas at the end.

A **CPU runtime is enough**: the baseline trains in under a minute. A GPU (*Runtime → Change runtime type → T4 GPU*)
helps for bigger models.

**The golden rule:** you **train** only on the train images and **choose** models and settings on the val images.
The official **test** images may only be used to *report* the results of your final models - never for training.

## Setup - run these two cells first
The first cell installs MedMNIST and loads DermaMNIST: 28 × 28 colour images of skin lesions, 7 classes.
The images become tensors `(N, 3, 28, 28)` with values 0...1, then they are **normalised** with the mean and
spread of the *training* images (one number per colour channel).

⚠️ Keep `MEAN`, `STD` and `normalize` exactly like this - the evaluation script and the app use the same numbers.

In [ ]:
!pip install -q medmnist
import os

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from medmnist import DermaMNIST
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import v2

MEAN = [0.7631, 0.5381, 0.5614]     # channel mean of the training images (R, G, B)
STD = [0.1366, 0.1543, 0.1692]      # channel spread of the training images
normalize = v2.Normalize(MEAN, STD)
CLASSES = ["actinic keratoses", "basal cell carcinoma", "benign keratosis", "dermatofibroma",
           "melanoma", "melanocytic nevi", "vascular lesions"]


def load(split):
    """DermaMNIST split as tensors: images (N, 3, 28, 28) normalised, labels (N,)."""
    os.makedirs("data", exist_ok=True)
    d = DermaMNIST(split=split, download=True, size=28, root="data")
    X = torch.tensor(d.imgs).permute(0, 3, 1, 2).float() / 255   # (N, 28, 28, 3) -> (N, 3, 28, 28), 0...1
    y = torch.tensor(d.labels).squeeze(1).long()
    return normalize(X), y


X_train, y_train = load("train")
X_val, y_val = load("val")
print("training:", tuple(X_train.shape), "  validation:", tuple(X_val.shape))

The second cell defines helpers for plots and metrics. **Just run it - you do not need to read it now.**
PyTorch has no built-in metrics, so we use scikit-learn **only for the metrics**.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             recall_score)


def show_examples(X, y, per_class=6):
    """A grid: one row per class, a few example images."""
    fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(per_class * 1.2, len(CLASSES) * 1.25))
    for c in range(len(CLASSES)):
        idx = (y == c).nonzero().squeeze(1)[:per_class]
        for k, ax in enumerate(axes[c]):
            img = X[idx[k]].permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)   # undo the normalisation
            ax.imshow(img.clamp(0, 1)); ax.axis("off")
        axes[c][0].set_title(CLASSES[c], fontsize=8, loc="left")
    plt.tight_layout(); plt.show()


def predict(model, X, batch_size=512):
    """Predicted class for every image (no gradients, eval mode)."""
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + batch_size]).argmax(1) for i in range(0, len(X), batch_size)])


def challenge_metrics(y_true, y_pred, show=True):
    """The challenge metrics. Main metric: balanced accuracy."""
    m = {"balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "accuracy": accuracy_score(y_true, y_pred),
         "macro_F1": f1_score(y_true, y_pred, average="macro")}
    if show:
        for k, v in m.items():
            print(f"{k:18s} {v:.3f}")
        sens = recall_score(y_true, y_pred, average=None, labels=range(len(CLASSES)), zero_division=0)
        print("sensitivity per class:")
        for name, s in zip(CLASSES, sens):
            print(f"   {name:22s} {s:.2f}")
    return m


def show_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    plt.figure(figsize=(5.5, 4.5))
    plt.imshow(cm, cmap="Blues")
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right", fontsize=8)
    plt.yticks(range(len(CLASSES)), CLASSES, fontsize=8)
    plt.xlabel("predicted"); plt.ylabel("true"); plt.tight_layout(); plt.show()


def train(model, epochs=10, lr=1e-3, class_weights=None, augment=False, seed=0):
    """The training loop from below, packed into a function - for your experiments."""
    torch.manual_seed(seed)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            if augment:                                   # random flips, a different one for every image
                flip = torch.rand(len(x)) < 0.5
                x = torch.where(flip[:, None, None, None], x.flip(-1), x)
                flip = torch.rand(len(x)) < 0.5
                x = torch.where(flip[:, None, None, None], x.flip(-2), x)
            loss = loss_fn(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


def bootstrap_difference(y_true, pred_a, pred_b, n_draws=2000, seed=0):
    """95 % interval of balanced_accuracy(B) - balanced_accuracy(A) on the same images (day 3)."""
    g = torch.Generator().manual_seed(seed)
    diffs = []
    for _ in range(n_draws):
        idx = torch.randint(0, len(y_true), (len(y_true),), generator=g)
        diffs.append(balanced_accuracy_score(y_true[idx], pred_b[idx])
                     - balanced_accuracy_score(y_true[idx], pred_a[idx]))
    low, high = torch.quantile(torch.tensor(diffs), torch.tensor([0.025, 0.975]))
    return round(low.item(), 3), round(high.item(), 3)


print("helpers ready")

## 1. Look at the data
**Always look at your data first.** How many images per class? What do they look like?

In [ ]:
counts = torch.bincount(y_train, minlength=len(CLASSES))
plt.figure(figsize=(7, 3))
plt.bar(CLASSES, counts)
plt.xticks(rotation=45, ha="right"); plt.ylabel("training images"); plt.show()
for name, n in zip(CLASSES, counts):
    print(f"{name:22s} {int(n):5d}  ({100 * n / len(y_train):.0f} %)")

show_examples(X_train, y_train)

❓ **Think:** two thirds of the images are *melanocytic nevi* (harmless moles). A model that always says
"nevus" gets about 67 % accuracy. That is why the **main metric is balanced accuracy** = the average
sensitivity over all 7 classes. What balanced accuracy does the "always nevus" model get?

<details><summary>👀 Answer</summary>

- sensitivity 100 % for nevi, 0 % for the 6 other classes → balanced accuracy = 100 / 7 ≈ **14 %**
- the counts (training): 228, 359, 769, 80, 779, **4693**, 99
</details>

## 2. The baseline CNN (= "A")
The CNN from the day-4 lab, now with **3 input channels** (colour) and **7 classes**.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 x 28 x 28  -> 16 x 28 x 28
    nn.ReLU(),
    nn.MaxPool2d(2),                              #              -> 16 x 14 x 14
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  #              -> 32 x 14 x 14
    nn.ReLU(),
    nn.MaxPool2d(2),                              #              -> 32 x 7 x 7
    nn.Flatten(),                                 #              -> 1568
    nn.Linear(32 * 7 * 7, len(CLASSES)),          #              -> 7 logits (one raw score per class)
)
print("parameters:", sum(p.numel() for p in model.parameters()))

The training loop - exactly the loop from day 3: forward, loss, `zero_grad`, `backward`, `step`.

In [ ]:
EPOCHS = 10      # 👈
LR = 1e-3        # 👈

loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()                                  # train mode
    for x, y in loader:
        loss = loss_fn(model(x), y)                # forward: how wrong are we?
        opt.zero_grad()
        loss.backward()                            # backward: gradients
        opt.step()                                 # update the weights
    val_pred = predict(model, X_val)               # eval mode + no_grad inside
    print("epoch", epoch + 1, "  last batch loss", round(loss.item(), 3),
          "  val balanced accuracy", round(balanced_accuracy_score(y_val, val_pred), 3))

## 3. Evaluate on the validation images

In [ ]:
val_pred = predict(model, X_val)
baseline_metrics = challenge_metrics(y_val, val_pred)
show_confusion(y_val, val_pred)

<details><summary>👀 What you should see (CPU: about 1 s per epoch)</summary>

- 16 071 parameters; the validation balanced accuracy climbs from **0.24** (epoch 1) to about **0.44** (epoch 10)
- accuracy **0.75**, but balanced accuracy only **0.44** and macro-F1 **0.45** - the model likes to say "nevus"
- sensitivity: nevi **0.91**, but dermatofibroma **0.00** and vascular lesions **0.29** - the rare classes are missed
- the confusion matrix: most mistakes end up in the "melanocytic nevi" column
- exact numbers change a little from run to run - that is why you use several seeds
</details>

## 4. Export the model - this is what you hand in
TorchScript saves the model *together with its code*: `model.pt` can be loaded without your notebook.
Your `model.pt` must take normalised images `(N, 3, 28, 28)` and return logits `(N, 7)`.

In [ ]:
model.eval()
scripted = torch.jit.script(model)          # turn the model into TorchScript
scripted.save("model.pt")                   # this file goes into your submission

loaded = torch.jit.load("model.pt")         # test: load it again, like the evaluation script does
same = torch.equal(predict(loaded, X_val), predict(model, X_val))
print("model.pt saved -", os.path.getsize("model.pt") // 1024, "KB - same predictions after loading:", same)
print("output shape for 2 images:", tuple(loaded(X_val[:2]).shape))

✅ You should see `same predictions after loading: True` and `output shape for 2 images: (2, 7)`.
Download `model.pt`: folder icon on the left in Colab → right-click → *Download*.

## 5. Your turn: improve the baseline (= "B")
Each experiment: change **one** thing, train with **3 seeds**, compare with the baseline (mean ± std), and
compute the 95 % interval of the difference (day 3). Some ideas:

| idea | how | why |
|---|---|---|
| class weights | `class_weights=` in `train(...)` (example below) | rare classes count more (day 3) |
| augmentation | `augment=True` (random flips) - or more: rotations, colour jitter | more variety, less overfitting |
| train longer / learning-rate schedule | `epochs=`, a scheduler (day 3) | the baseline is not finished after 10 epochs |
| a bigger CNN | more filters, a third conv block, BatchNorm, dropout | more capacity |
| a pretrained ResNet | day 5: transfer learning | features learned on millions of images |

Find at least one idea in a paper or a good source and cite it in your report.

📈 **Track your experiments** with one of these free tools - then no result gets lost, and a screenshot of your
runs fits nicely into the report (day 3, "Keep track of your runs"):

| tool | what you get | how to start |
|---|---|---|
| [Weights & Biases](https://wandb.ai) | online dashboard, compare runs, sweeps | free (academic) account; `wandb.init(project=...)`, `wandb.log({...})` |
| [MLflow](https://mlflow.org) | open source, runs stored in a local folder or on a server | `mlflow.start_run()`, `mlflow.log_param(...)`, `mlflow.log_metric(...)` |
| [ClearML](https://clear.ml) | open source, free hosted tier, logs PyTorch runs automatically | free account; `Task.init(project_name=..., task_name=...)` |

In [ ]:
CLASS_WEIGHTS = True   # 👈 True: weight = (number of images) / (7 x images of this class)
AUGMENT = False        # 👈
SEEDS = [0, 1, 2]      # 👈 at least 3 seeds


def make_baseline():
    return nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(32 * 7 * 7, len(CLASSES)))


weights = len(y_train) / (len(CLASSES) * counts.float()) if CLASS_WEIGHTS else None
results_a, results_b = [], []
for seed in SEEDS:
    torch.manual_seed(seed)
    pred_a = predict(train(make_baseline(), seed=seed), X_val)                              # A: baseline
    torch.manual_seed(seed)
    pred_b = predict(train(make_baseline(), class_weights=weights, augment=AUGMENT, seed=seed), X_val)  # B
    results_a.append(balanced_accuracy_score(y_val, pred_a))
    results_b.append(balanced_accuracy_score(y_val, pred_b))
    print("seed", seed, "  A:", round(results_a[-1], 3), "  B:", round(results_b[-1], 3),
          "  95 % interval of B - A:", bootstrap_difference(y_val, pred_a, pred_b))

a, b = torch.tensor(results_a), torch.tensor(results_b)
print("A: mean", round(a.mean().item(), 3), "± std", round(a.std().item(), 3))
print("B: mean", round(b.mean().item(), 3), "± std", round(b.std().item(), 3))

<details><summary>👀 What you should see (class weights, no augmentation, about 1 minute on CPU)</summary>

- A (baseline): balanced accuracy **0.46 ± 0.02** (seeds: 0.48, 0.47, 0.44)
- B (class weights): **0.53 ± 0.01** (0.53, 0.54, 0.52) - the rare classes are found more often
- 95 % intervals of B - A: seed 0 **[-0.005, 0.109]**, seed 1 **[0.018, 0.126]**, seed 2 **[0.022, 0.143]** -
  two of three lie above 0: probably a real improvement, but one seed alone would not have shown it clearly
- one validation set of 1 003 images is small - that is why you report several seeds *and* the interval
</details>

**Do not forget:** when you are happy with your model, export it again (section 4) and hand in the new `model.pt`.